In [1]:
import os
import sys
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import pickle
import seaborn as sns

# 0. Define the specific ablation environment to visualize
dataset_folder = 'preprocessed_proposal'
model_name = 'cnn'
target_tput_mbps = 0.1
window_size = 16
constraint_type = 'AR'
experiment_name = f'ablation_win{window_size}_kdeTrue_{constraint_type}_seed21'

# Broadcast environment variables so MLConfig loads correctly
os.environ['ML_WINDOW_SIZE'] = str(window_size)
os.environ['ML_USE_KDE'] = 'True'
os.environ['ML_CONSTRAINT_TYPE'] = constraint_type
os.environ['ML_SEED'] = '21'
os.environ['ML_EXPERIMENT_NAME'] = experiment_name

# Add the parent directory to the path to import custom modules
scripts_path = os.path.abspath('../scripts/deep_learning_4')
if scripts_path not in sys.path:
    sys.path.append(scripts_path)

scripts_root = os.path.abspath('../scripts')
if scripts_root not in sys.path:
    sys.path.append(scripts_root)
    
from online_selector import OnlineModeSelector
from ml_config import MLConfig

# Recreate the windowing function for the notebook
def create_sliding_windows_for_episode(X, y, win_size=MLConfig.WINDOW_SIZE):
    X_win, y_win = [], []
    for t in range(X.shape[0] - win_size + 1):
        X_win.append(X[t:t+win_size, :])
        y_win.append(y[t+win_size-1, :])
    return np.array(X_win), np.array(y_win)

# 1. Load Data (Using the test set from the specific dataset)
X_test_d2d = np.load(f"../data/{dataset_folder}/d2d/X_test.npy")
y_test_d2d = np.load(f"../data/{dataset_folder}/d2d/y_test.npy")
X_test_cell = np.load(f"../data/{dataset_folder}/cellular/X_test.npy")
y_test_cell = np.load(f"../data/{dataset_folder}/cellular/y_test.npy")

# Select a single random episode (e.g., Episode 0) to visualize
ep_idx = 0 
X_ep_d2d, y_ep_d2d = create_sliding_windows_for_episode(X_test_d2d[ep_idx], y_test_d2d[ep_idx])
X_ep_cell, y_ep_cell = create_sliding_windows_for_episode(X_test_cell[ep_idx], y_test_cell[ep_idx])

# 2. Load the DL Models, Scalers, and Error Params
model_dir_d2d = f"../models/{dataset_folder}/{experiment_name}/d2d/{model_name}"
model_dir_cell = f"../models/{dataset_folder}/{experiment_name}/cellular/{model_name}"

model_d2d = tf.keras.models.load_model(f"{model_dir_d2d}/{model_name}_model.keras")
model_cell = tf.keras.models.load_model(f"{model_dir_cell}/{model_name}_model.keras")

# Load Target Scalers to convert predictions back to dB
with open(f"../data/{dataset_folder}/d2d/target_scaler.pkl", "rb") as f:
    scaler_d2d = pickle.load(f)
with open(f"../data/{dataset_folder}/cellular/target_scaler.pkl", "rb") as f:
    scaler_cell = pickle.load(f)

# Load Error Parameters for plotting bounds (Already in dB from error_analysis script)
with open(f"{model_dir_d2d}/{model_name}_error_params_kde.pkl", "rb") as f:
    err_params_d2d = pickle.load(f)
with open(f"{model_dir_cell}/{model_name}_error_params_kde.pkl", "rb") as f:
    err_params_cell = pickle.load(f)

# 3. Generate Predictions and Inverse Transform to raw dB
preds_d2d_z = model_d2d.predict(X_ep_d2d, verbose=0)
preds_cell_z = model_cell.predict(X_ep_cell, verbose=0)

preds_d2d_db = scaler_d2d.inverse_transform(preds_d2d_z).flatten()
preds_cell_db = scaler_cell.inverse_transform(preds_cell_z).flatten()
true_d2d_db = scaler_d2d.inverse_transform(y_ep_d2d).flatten()
true_cell_db = scaler_cell.inverse_transform(y_ep_cell).flatten()

# 4. Simulate the Online Selector step-by-step
selector = OnlineModeSelector(
    model_name=model_name, 
    dataset_folder=dataset_folder, 
    constraint_type=constraint_type, 
    target_tput_mbps=target_tput_mbps
)
current_mode = 'D2D'

# Arrays to store the data for plotting
time_steps = []
true_sinr_log = []
pred_sinr_log = []
upper_bound_log = []
lower_bound_log = []
throughput_log = []
switch_points_x = []
switch_points_y = []

print(f"\nSimulating 1 Episode using {model_name.upper()}...")
for t in range(len(preds_d2d_db)):
    # Physical ground truth in dB
    true_d2d = true_d2d_db[t]
    true_cell = true_cell_db[t]
    pred_d2d = preds_d2d_db[t]
    pred_cell = preds_cell_db[t]
    
    # Make decision
    new_mode, logs = selector.make_decision(pred_d2d, pred_cell, current_mode)
    
    # Track switches
    if new_mode != current_mode:
        switch_points_x.append(t)
        
    current_mode = new_mode
    
    # Calculate actual realized throughput based on the physical signal of the chosen mode
    if current_mode == 'D2D':
        actual_tput = selector.ts.shannon_throughput(true_d2d)
        active_true_sinr = true_d2d
        active_pred_sinr = pred_d2d
        margin_upper = err_params_d2d['upper_bound']
        margin_lower = err_params_d2d['lower_bound']
    else:
        actual_tput = selector.ts.shannon_throughput(true_cell)
        active_true_sinr = true_cell
        active_pred_sinr = pred_cell
        margin_upper = err_params_cell['upper_bound'] 
        margin_lower = err_params_cell['lower_bound']
        
    # Append the Y-value (throughput) for the scatter plot if a switch just happened
    if t in switch_points_x and len(switch_points_y) < len(switch_points_x):
        switch_points_y.append(actual_tput)

    # Log data
    time_steps.append(t)
    true_sinr_log.append(active_true_sinr)
    pred_sinr_log.append(active_pred_sinr)
    upper_bound_log.append(active_pred_sinr + margin_upper)
    lower_bound_log.append(active_pred_sinr + margin_lower)
    throughput_log.append(actual_tput)

print("Simulation Complete. Ready to plot!")

⚠️ No GPU found. Training on CPU. (Running Seed: 21)


ValueError: File not found: filepath=../models/preprocessed_proposal/ablation_win16_kdeTrue_AR_seed21/d2d/cnn/cnn_model.keras. Please ensure the file is an accessible `.keras` zip file.

In [ ]:
# Set plain background like the paper
sns.set_style("ticks")
plt.figure(figsize=(16, 8))

# Plot the 3 lines
plt.plot(time_steps, true_sinr_log, color='blue', linewidth=2, label='Real', alpha=0.8)
plt.plot(time_steps, pred_sinr_log, color='red', linewidth=2, label='Predicted', alpha=0.8)
plt.plot(time_steps, upper_bound_log, color='orange', linewidth=3, label='Upper and lower bounds')
plt.plot(time_steps, lower_bound_log, color='orange', linewidth=3)

# Formatting to match the research paper style
plt.xlim(0, len(time_steps))
plt.xlabel('Time Interval', fontsize=16, fontweight='bold')
plt.ylabel('SINR (dB)', fontsize=16, fontweight='bold')

# Remove top and right borders (spines) to match the uploaded graph
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.legend(loc='upper right', fontsize=12)

# Save and show
os.makedirs(f'../results/{dataset_folder}/{experiment_name}/system_viz', exist_ok=True)
plt.savefig(f'../results/{dataset_folder}/{experiment_name}/system_viz/sinr_confidence_intervals.png', dpi=300, bbox_inches='tight')
plt.show()

NameError: name 'time_steps' is not defined

<Figure size 1600x800 with 0 Axes>

In [ ]:
plt.figure(figsize=(16, 8))

# Plot the throughput line
plt.plot(time_steps, throughput_log, color='#1f77b4', linewidth=2)

# Scatter plot the red dots where a mode switch occurred
plt.scatter(switch_points_x, switch_points_y, color='red', s=80, zorder=5, label='Mode Switch')

# Formatting to match the research paper style
plt.xlim(0, len(time_steps))
plt.ylim(0, max(throughput_log) * 1.1)
plt.xlabel('Time Interval', fontsize=14)
plt.ylabel('Throughput (Mbps)', fontsize=14)

# Remove top and right borders 
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.legend(loc='lower right', fontsize=14, frameon=True, shadow=True)

# Save and show
plt.savefig(f'../results/{dataset_folder}/{experiment_name}/system_viz/throughput_mode_switches.png', dpi=300, bbox_inches='tight')
plt.show()